# PyTorch nn.Module 实验课

像搭乐高一样搭神经网络。配合 `tutorial/03_nn_module.py` 学习。

In [4]:
import torch
import torch.nn as nn

print(f"PyTorch version: {torch.__version__}")

PyTorch version: 2.12.1


## 1. 最简单的 nn.Module — 继承 + __init__ + forward

In [6]:
class SimpleNet(nn.Module):
    def __init__(self):
        super().__init__()
        # 把层定义为 self 的属性，nn.Module 自动发现它们
        self.fc1 = nn.Linear(10, 20)   # 输入10维 → 输出20维
        self.fc2 = nn.Linear(20, 5)    # 输入20维 → 输出5维
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

model = SimpleNet()
print(model)
print(f"\nfc1.weight shape: {model.fc1.weight.shape}")  # (20, 10)
print(f"fc1.bias shape:   {model.fc1.bias.shape}")      # (20,)

SimpleNet(
  (fc1): Linear(in_features=10, out_features=20, bias=True)
  (fc2): Linear(in_features=20, out_features=5, bias=True)
  (relu): ReLU()
)

fc1.weight shape: torch.Size([20, 10])
fc1.bias shape:   torch.Size([20])


## 2. 前向传播 — model(x) 等价于 model.forward(x)

In [7]:
x = torch.randn(3, 10)   # batch_size=3, input_dim=10
out = model(x)
print(f"输入 shape: {x.shape}")
print(f"输出 shape: {out.shape}   # (3, 5)，每行是一个样本的预测")

# ⚠️ 永远不要直接调 model.forward(x)！
# 调 model(x) 会走 __call__，额外处理 hooks 等逻辑

输入 shape: torch.Size([3, 10])
输出 shape: torch.Size([3, 5])   # (3, 5)，每行是一个样本的预测


## 3. 常用层一览

In [8]:
# --- 线性层（全连接）---
linear = nn.Linear(in_features=4, out_features=3)
print(f"nn.Linear(4, 3): weight={linear.weight.shape}, bias={linear.bias.shape}")

# --- 激活函数 ---
print(f"nn.ReLU()     : max(0, x)")
print(f"nn.Sigmoid()  : 1/(1+e^x)  → 压缩到 (0,1)")
print(f"nn.Tanh()     : 双曲正切 → 压缩到 (-1,1)")
print(f"nn.Softmax(dim=1): e^x/Σe^x  → 概率分布")

# 演示 softmax
logits = torch.tensor([[2.0, 1.0, 0.1]])
print(f"\nlogits = {logits}")
print(f"Softmax = {torch.softmax(logits, dim=1)}   # 变成概率，和为1")

nn.Linear(4, 3): weight=torch.Size([3, 4]), bias=torch.Size([3])
nn.ReLU()     : max(0, x)
nn.Sigmoid()  : 1/(1+e^x)  → 压缩到 (0,1)
nn.Tanh()     : 双曲正切 → 压缩到 (-1,1)
nn.Softmax(dim=1): e^x/Σe^x  → 概率分布

logits = tensor([[2.0000, 1.0000, 0.1000]])
Softmax = tensor([[0.6590, 0.2424, 0.0986]])   # 变成概率，和为1


In [9]:
# --- Dropout（防过拟合）---
dropout = nn.Dropout(p=0.5)
dropout.train()
x = torch.ones(1, 6)
print(f"Dropout.train(): {dropout(x)}  # ~一半是0，其余放大")

dropout.eval()
print(f"Dropout.eval():  {dropout(x)}  # 原样输出")

# --- BatchNorm（批标准化）---
bn = nn.BatchNorm1d(num_features=4)
x = torch.randn(3, 4) * 10  # 大数值
print(f"\nBatchNorm 前: mean={x.mean().item():.2f}, std={x.std().item():.2f}")
print(f"BatchNorm 后: mean={bn(x).mean().item():.2f}, std={bn(x).std().item():.2f}")

Dropout.train(): tensor([[2., 0., 0., 2., 0., 0.]])  # ~一半是0，其余放大
Dropout.eval():  tensor([[1., 1., 1., 1., 1., 1.]])  # 原样输出

BatchNorm 前: mean=-0.27, std=12.62
BatchNorm 后: mean=-0.00, std=1.04


## 4. nn.Sequential — 不用写 forward 的搭积木式写法

In [10]:
# Sequential：层按顺序执行，不需要手动写 forward
seq_net = nn.Sequential(
    nn.Linear(10, 20),
    nn.ReLU(),
    nn.Linear(20, 5),
)

x = torch.randn(3, 10)
out = seq_net(x)
print(f"Sequential 网络:\n{seq_net}")
print(f"\n输入 {x.shape} → 输出 {out.shape}")

# 对比手写版本：用 SimpleNet 做同样的事
model = SimpleNet()
out2 = model(x)
print(f"\n手写版输出 shape: {out2.shape}  # 结果一样")

Sequential 网络:
Sequential(
  (0): Linear(in_features=10, out_features=20, bias=True)
  (1): ReLU()
  (2): Linear(in_features=20, out_features=5, bias=True)
)

输入 torch.Size([3, 10]) → 输出 torch.Size([3, 5])

手写版输出 shape: torch.Size([3, 5])  # 结果一样


## 5. parameters() — 自动收集所有可训练参数

In [17]:
model = SimpleNet()
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"总参数数量: {total_params:,}")
print(f"可训练参数: {trainable_params:,}")

# 逐层查看参数
print("\n逐层参数：")
for name, param in model.named_parameters():
    print(f"  {name:25s}  shape={str(param.shape):15s}  requires_grad={param.requires_grad}")

总参数数量: 325
可训练参数: 325

逐层参数：
  fc1.weight                 shape=torch.Size([20, 10])  requires_grad=True
  fc1.bias                   shape=torch.Size([20])  requires_grad=True
  fc2.weight                 shape=torch.Size([5, 20])  requires_grad=True
  fc2.bias                   shape=torch.Size([5])  requires_grad=True


## 6. train() vs eval() — Dropout / BatchNorm 的行为差异

In [20]:
model = nn.Sequential(
    nn.Linear(4, 4),
    nn.BatchNorm1d(4),
    nn.Dropout(0.5),
)

x = torch.randn(2, 4)

# 训练模式
model.train()
out_train = model(x)
print(f"train 模式输出: {out_train}")
print(f"  Dropout 生效，BatchNorm 用当前 batch 统计量")

# 评估模式
model.eval()
with torch.no_grad():
    out_eval = model(x)
print(f"eval  模式输出: {out_eval}")
print(f"  Dropout 关闭，BatchNorm 用训练时的 running mean/var")

print(f"\n两种模式结果不同: {not torch.allclose(out_train, out_eval)}")

train 模式输出: tensor([[ 1.9998, -2.0000, -0.0000,  1.9998],
        [-1.9998,  2.0000,  1.9999, -0.0000]], grad_fn=<MulBackward0>)
  Dropout 生效，BatchNorm 用当前 batch 统计量
eval  模式输出: tensor([[ 0.7968, -0.5836,  0.0316, -0.0501],
        [ 0.3660,  0.8775,  0.9435, -0.5810]])
  Dropout 关闭，BatchNorm 用训练时的 running mean/var

两种模式结果不同: True


## 7. 自定义参数 — nn.Parameter

In [22]:
class LearnableScale(nn.Module):
    """y = scale * x + bias，scale 和 bias 都是可训练参数"""
    def __init__(self, dim: int):
        super().__init__()
        self.scale = nn.Parameter(torch.ones(dim))
        self.bias = nn.Parameter(torch.zeros(dim))

    def forward(self, x):
        return self.scale * x + self.bias

layer = LearnableScale(5)
x = torch.randn(3, 5)
out = layer(x)
print(f"LearnableScale: {x.shape} → {out.shape}")

params = list(layer.parameters())
print(f"参数数量: {len(params)}  # scale + bias")
print(f"scale: {params[0]}")
print(f"bias:  {params[1]}")

LearnableScale: torch.Size([3, 5]) → torch.Size([3, 5])
参数数量: 2  # scale + bias
scale: Parameter containing:
tensor([1., 1., 1., 1., 1.], requires_grad=True)
bias:  Parameter containing:
tensor([0., 0., 0., 0., 0.], requires_grad=True)


## 8. 完整示例 — MLP 多层感知机

In [18]:
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dims, output_dim, dropout=0.2):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for h_dim in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, h_dim),
                nn.BatchNorm1d(h_dim),
                nn.ReLU(),
                nn.Dropout(dropout),
            ])
            prev_dim = h_dim
        layers.append(nn.Linear(prev_dim, output_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

mlp = MLP(input_dim=100, hidden_dims=[64, 32], output_dim=10, dropout=0.3)
x = torch.randn(16, 100)
out = mlp(x)
print(f"MLP(100→64→32→10): 输入 {x.shape} → 输出 {out.shape}")
print(f"总参数: {sum(p.numel() for p in mlp.parameters()):,}")

MLP(100→64→32→10): 输入 torch.Size([16, 100]) → 输出 torch.Size([16, 10])
总参数: 9,066


## 9. 残差连接 — Residual Block（Transformer 的核心模式）

In [21]:
class ResidualBlock(nn.Module):
    """out = ReLU(Linear(ReLU(Linear(x))) + x)"""
    def __init__(self, dim: int):
        super().__init__()
        self.linear1 = nn.Linear(dim, dim)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(dim, dim)

    def forward(self, x):
        out = self.linear1(x)
        out = self.relu(out)
        out = self.linear2(out)
        return self.relu(out + x)  # ← 残差连接！跳过两层

block = ResidualBlock(16)
x = torch.randn(4, 16)
out = block(x)
print(f"ResidualBlock(16): 输入 {x.shape} → 输出 {out.shape}")
print(f"输入输出形状一致 ✓  (因为 dim 保持不变)")

ResidualBlock(16): 输入 torch.Size([4, 16]) → 输出 torch.Size([4, 16])
输入输出形状一致 ✓  (因为 dim 保持不变)


## 📝 核心速查

| 操作 | 说明 |
|------|------|
| `class MyNet(nn.Module)` | 继承基类 |
| `super().__init__()` | 必须调用父类构造 |
| `self.fc = nn.Linear(in, out)` | 定义层 |
| `def forward(self, x)` | 定义前向传播 |
| `model(x)` | 前向传播（不要调 .forward） |
| `nn.Sequential(*layers)` | 顺序组合，不用写 forward |
| `model.parameters()` | 自动收集所有可训练参数 |
| `nn.Parameter(tensor)` | 包裹成可训练参数 |
| `model.train()` / `.eval()` | 切换训练/评估模式 |
| `torch.no_grad():` | 推理时关闭梯度追踪 |

**残差连接（Transformer 基石）：**
```python
def forward(self, x):
    return self.relu(self.linear2(self.relu(self.linear1(x))) + x)
```

👉 下一步：去 `practice/03_network_building.py` 刷题！

In [16]:
# 来，看看矩阵长啥样
linear = nn.Linear(4, 3)  # 输入4维 → 输出3维

print("权重矩阵 W (shape 3×4):")
print(linear.weight)
print(f"\n偏置 b (shape 3):")
print(linear.bias)

print("\n每一行对应一个输出神经元，每一列对应一个输入特征")
print("输出 y₁ = x₁·w₁₁ + x₂·w₁₂ + x₃·w₁₃ + x₄·w₁₄ + b₁")
print("输出 y₂ = x₁·w₂₁ + x₂·w₂₂ + x₃·w₂₃ + x₄·w₂₄ + b₂")
print("输出 y₃ = x₁·w₃₁ + x₂·w₃₂ + x₃·w₃₃ + x₄·w₃₄ + b₃")

# 演示一下实际计算
x = torch.tensor([1.0, 2.0, 3.0, 4.0])
y = linear(x)
print(f"\n输入 x = {x}")
print(f"输出 y = x@W.T + b = {y}")
print(f"手动验证 y₁ = {x.dot(linear.weight[0]) + linear.bias[0]:.4f}  ✓")

权重矩阵 W (shape 3×4):
Parameter containing:
tensor([[-0.1144,  0.2231,  0.1802, -0.4527],
        [-0.0210,  0.1217, -0.3734,  0.3757],
        [ 0.1416, -0.0612, -0.1813, -0.0603]], requires_grad=True)

偏置 b (shape 3):
Parameter containing:
tensor([0.2450, 0.1365, 0.3540], requires_grad=True)

每一行对应一个输出神经元，每一列对应一个输入特征
输出 y₁ = x₁·w₁₁ + x₂·w₁₂ + x₃·w₁₃ + x₄·w₁₄ + b₁
输出 y₂ = x₁·w₂₁ + x₂·w₂₂ + x₃·w₂₃ + x₄·w₂₄ + b₂
输出 y₃ = x₁·w₃₁ + x₂·w₃₂ + x₃·w₃₃ + x₄·w₃₄ + b₃

输入 x = tensor([1., 2., 3., 4.])
输出 y = x@W.T + b = tensor([-0.6934,  0.7415, -0.4120], grad_fn=<ViewBackward0>)
手动验证 y₁ = -0.6934  ✓
